# RealSense 3D Depth Analysis
Visualizing depth maps and interactive 3D topography for each run within selected ROI

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import plotly.graph_objects as go
import plotly.io as pio
from matplotlib.patches import Rectangle

pio.renderers.default = 'notebook'

INVERT_DEPTH         = False
MIN_DEPTH_MM         = 600
MAX_DEPTH_MM         = 2000
SPLAT_FILL           = True
AVERAGE_NONZERO_ONLY = True

ULTRASONIC_CM = {
    'run1':  12.95, 'run2':  10.75, 'run3':  15.3,  'run4':  22.78,
    'run5':  36.45, 'run6':   8.55, 'run7':  11.05, 'run8':  11.55,
    'run9':  10.45, 'run10': 30.0,  'run11': 37.3,  'run12': 19.65,
    'run13': 27.75, 'run14': 25.05, 'run15': 21.5,
    'run16': 19.3,  'run17': 30.1,  'run18': 14.3,  'run19': 30.5,
    'run20': 32.3,  'run21':  4.2,  'run22':  6.1,  'run23': 34.9,
    'run24': 20.9,  'run25': 34.3,  'run26': 27.5,  'run27': 19.9,
    'run28':  4.1,  'run29':  9.1,  'run30': 34.7,  'run31': 27.9,
    'run32': 20.1,  'run33': 27.1,  'run34': 19.6,  'run35': 18.8,
    'run36': 32.3,  'run37': 20.1,  'run38': 21.0,  'run39': 23.4,
    'run40': 14.1,  'run41': 30.3,  'run42': 27.5,  'run43': 21.3,
    'run44': 19.4,  'run45': 13.5,  'run46': 33.3,  'run47': 34.3,
    'run48': 27.9,  'run49': 26.1,  'run50':  4.9,
}

# Group A: run1-15  -> 676 mm
# Group B: run16    -> 731 mm
# Group C: run17-50 -> 780 mm
def get_base_depth(run_name):
    n = int(run_name.replace('run', ''))
    if n <= 15:  return 676
    if n == 16:  return 731
    return 780

analysis_dir = Path(r'C:\Users\Umer\Downloads\Cleanify Depth Analysis 2\analysis')
run_dir      = Path(r'C:\Users\Umer\Downloads\Cleanify Depth Analysis 2\run')

with open(analysis_dir / 'rectangle_coordinates.json', 'r') as f:
    coordinates = json.load(f)

print(f"Loaded coordinates for {len(coordinates)} runs")
print(f"Valid depth range: {MIN_DEPTH_MM} - {MAX_DEPTH_MM} mm")

In [ ]:
import pyrealsense2 as rs
import cv2
import numpy as np
from scipy.interpolate import LinearNDInterpolator, NearestNDInterpolator


def extract_depth_and_color_frames(bag_path):
    depth_frames, color_frames = [], []
    config = rs.config()
    config.enable_device_from_file(str(bag_path), repeat_playback=False)
    pipeline = rs.pipeline()
    pipeline.start(config)
    while True:
        try:
            frames = pipeline.wait_for_frames(timeout_ms=1000)
            df = frames.get_depth_frame()
            cf = frames.get_color_frame()
            if df and cf:
                depth_frames.append(np.asanyarray(df.get_data()))
                color_frames.append(cv2.cvtColor(np.asanyarray(cf.get_data()), cv2.COLOR_BGR2RGB))
        except RuntimeError:
            break
    pipeline.stop()
    return depth_frames, color_frames


def compute_average_depth(depth_frames, average_nonzero_only=True):
    frames = np.array(depth_frames, dtype=np.float32)
    if average_nonzero_only:
        valid = frames > 0
        total = np.sum(np.where(valid, frames, 0.0), axis=0)
        count = np.sum(valid, axis=0)
        return np.divide(total, count, where=count > 0, out=np.zeros_like(total))
    return np.mean(frames, axis=0)


def roi_to_rect(roi):
    if isinstance(roi, list) and len(roi) == 4 and isinstance(roi[0], (list, tuple)):
        xs = [p[0] for p in roi]
        ys = [p[1] for p in roi]
        return min(xs), min(ys), max(xs), max(ys)
    return roi[0], roi[1], roi[2], roi[3]


def sort_quad_points(pts):
    pts = np.array(pts, dtype=np.float32)
    s = pts[:, 0] + pts[:, 1]
    d = pts[:, 0] - pts[:, 1]
    tl = pts[np.argmin(s)]
    br = pts[np.argmax(s)]
    tr = pts[np.argmax(d)]
    bl = pts[np.argmin(d)]
    return np.array([tl, tr, br, bl], dtype=np.float32)


def bbox_crop(img, roi):
    x1, y1, x2, y2 = roi_to_rect(roi)
    return img[y1:y2, x1:x2]


def perspective_warp(img, roi):
    if not (isinstance(roi, list) and len(roi) == 4 and isinstance(roi[0], (list, tuple))):
        return img
    x1, y1, _, _ = roi_to_rect(roi)
    local_roi = [[p[0]-x1, p[1]-y1] for p in roi]
    src = sort_quad_points(local_roi)
    tl, tr, br, bl = src
    w = int((np.linalg.norm(tr - tl) + np.linalg.norm(br - bl)) / 2)
    h = int((np.linalg.norm(bl - tl) + np.linalg.norm(br - tr)) / 2)
    dst = np.array([[0,0],[w-1,0],[w-1,h-1],[0,h-1]], dtype=np.float32)
    M   = cv2.getPerspectiveTransform(src, dst)
    bv  = 0.0 if img.ndim == 2 else (0,0,0)
    return cv2.warpPerspective(img, M, (w, h), flags=cv2.INTER_LINEAR,
                               borderMode=cv2.BORDER_CONSTANT, borderValue=bv)


def crop_to_roi(img, roi):
    return perspective_warp(bbox_crop(img, roi), roi)


def get_warp_valid_mask(roi, bbox_shape):
    if not (isinstance(roi, list) and len(roi) == 4 and isinstance(roi[0], (list, tuple))):
        return None
    x1, y1, _, _ = roi_to_rect(roi)
    local_roi = [[p[0]-x1, p[1]-y1] for p in roi]
    src = sort_quad_points(local_roi)
    tl, tr, br, bl = src
    w = int((np.linalg.norm(tr - tl) + np.linalg.norm(br - bl)) / 2)
    h = int((np.linalg.norm(bl - tl) + np.linalg.norm(br - tr)) / 2)
    dst = np.array([[0,0],[w-1,0],[w-1,h-1],[0,h-1]], dtype=np.float32)
    M   = cv2.getPerspectiveTransform(src, dst)
    ones = np.ones(bbox_shape[:2], dtype=np.uint8) * 255
    warped = cv2.warpPerspective(ones, M, (w, h), flags=cv2.INTER_NEAREST,
                                 borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    return warped > 128


def splat_fill(depth):
    h, w = depth.shape
    y_coords, x_coords = np.mgrid[0:h, 0:w]
    valid = ~np.isnan(depth)
    if not np.any(~valid):
        return depth
    pts   = np.column_stack([x_coords[valid],  y_coords[valid]])
    vals  = depth[valid]
    holes = np.column_stack([x_coords[~valid], y_coords[~valid]])
    filled = LinearNDInterpolator(pts, vals)(holes)
    nan_mask = np.isnan(filled)
    if np.any(nan_mask):
        filled[nan_mask] = NearestNDInterpolator(pts, vals)(holes[nan_mask])
    result = depth.copy()
    result[~valid] = filled
    return result


def find_wall_mask(run_name, crop_shape, depth_roi, analysis_dir):
    mask_path = analysis_dir / "Wall Masks" / f"{run_name}_depth_avg.png"
    if not mask_path.exists():
        return np.ones(crop_shape, dtype=bool)
    mask_full = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask_full is None:
        return np.ones(crop_shape, dtype=bool)
    mask_warped = crop_to_roi(mask_full, depth_roi)
    if mask_warped.shape != crop_shape:
        mask_warped = cv2.resize(mask_warped, (crop_shape[1], crop_shape[0]),
                                 interpolation=cv2.INTER_NEAREST)
    return mask_warped > 0



def perspective_warp_nearest(img, roi):
    if not (isinstance(roi, list) and len(roi) == 4 and isinstance(roi[0], (list, tuple))):
        return img
    x1, y1, _, _ = roi_to_rect(roi)
    local_roi = [[p[0]-x1, p[1]-y1] for p in roi]
    src_pts = sort_quad_points(local_roi)
    tl, tr, br, bl = src_pts
    w = int((np.linalg.norm(tr - tl) + np.linalg.norm(br - bl)) / 2)
    h = int((np.linalg.norm(bl - tl) + np.linalg.norm(br - tr)) / 2)
    dst = np.array([[0,0],[w-1,0],[w-1,h-1],[0,h-1]], dtype=np.float32)
    M   = cv2.getPerspectiveTransform(src_pts, dst)
    bv  = 0.0 if img.ndim == 2 else (0,0,0)
    return cv2.warpPerspective(img, M, (w, h), flags=cv2.INTER_NEAREST,
                               borderMode=cv2.BORDER_CONSTANT, borderValue=bv)

global_max_depth = 563.2
global_max_w     = 225
global_max_h     = 208

print("Helper functions loaded.")

In [ ]:
global_max_depth = 0.0
global_max_w = 0
global_max_h = 0

for run_name, coord in coordinates.items():
    bag_file = run_dir / f'{run_name}.bag'
    if not bag_file.exists():
        continue
    if 'depth' not in coord:
        continue

    depth_roi = coord['depth']
    x1, y1, x2, y2 = roi_to_rect(depth_roi)
    roi_w = x2 - x1
    roi_h = y2 - y1
    if roi_w > global_max_w:
        global_max_w = roi_w
    if roi_h > global_max_h:
        global_max_h = roi_h

    try:
        depth_frames, _ = extract_depth_and_color_frames(bag_file)
        avg_depth  = compute_average_depth(depth_frames, average_nonzero_only=AVERAGE_NONZERO_ONLY)
        depth_crop = crop_to_roi(avg_depth, depth_roi)

        depth_nan = depth_crop.astype(np.float32)
        depth_nan[(depth_nan == 0) |
                  (depth_nan < MIN_DEPTH_MM) |
                  (depth_nan > MAX_DEPTH_MM)] = np.nan

        inner_mask = find_wall_mask(run_name, depth_nan.shape, depth_roi, analysis_dir)
        depth_nan[~inner_mask] = np.nan

        filled = splat_fill(depth_nan) if SPLAT_FILL else depth_nan

        run_min = np.nanmin(filled)
        shifted = filled - run_min
        run_max = np.nanmax(shifted)

        if run_max > global_max_depth:
            global_max_depth = run_max

        print(f"  {run_name}: shifted max = {run_max:.1f} mm | ROI = {roi_w}x{roi_h} px")

    except Exception as e:
        print(f"  {run_name}: SKIPPED — {e}")

print(f"\nGLOBAL_DEPTH_MAX  = {global_max_depth:.1f} mm")
print(f"GLOBAL_MAX_W      = {global_max_w} px")
print(f"GLOBAL_MAX_H      = {global_max_h} px")

In [138]:
def plot_run_analysis(run_name, coordinates, run_dir):
    BASE_DEPTH_MM = get_base_depth(run_name)
    bag_file = run_dir / f"{run_name}.bag"
    if run_name not in coordinates or "depth" not in coordinates[run_name]:
        print(f"No depth ROI for {run_name}")
        return

    depth_roi = coordinates[run_name]["depth"]
    rgb_roi   = coordinates[run_name].get("rgb", depth_roi)

    ultrasonic_cm = ULTRASONIC_CM.get(run_name, None)
    ultrasonic_level_mm = BASE_DEPTH_MM + (ultrasonic_cm * 10) if ultrasonic_cm is not None else None

    # Apply linear correction (scale+bias) if enabled after running the bias cell
    _use_corr = "CORRECTION_PARAMS" in globals() and USE_CORRECTION and run_name in CORRECTION_PARAMS
    if _use_corr and ultrasonic_level_mm is not None:
        _sc, _bi = CORRECTION_PARAMS[run_name]
        ultrasonic_level_mm = _sc * ultrasonic_level_mm + _bi

    try:
        print(f"Processing {run_name}...")
        depth_frames, color_frames = extract_depth_and_color_frames(bag_file)
        avg_depth = compute_average_depth(depth_frames, average_nonzero_only=AVERAGE_NONZERO_ONLY)

        # 1. bbox crop and filter invalids
        depth_bbox    = bbox_crop(avg_depth, depth_roi)
        depth_raw_nan = depth_bbox.astype(np.float32)
        depth_raw_nan[(depth_raw_nan == 0) |
                      (depth_raw_nan < MIN_DEPTH_MM) |
                      (depth_raw_nan > MAX_DEPTH_MM)] = np.nan

        # 2. splat fill in bbox space (no border gaps present yet)
        depth_filled_bbox = splat_fill(depth_raw_nan) if SPLAT_FILL else depth_raw_nan

        # 3. INTER_NEAREST warp — border pixels are exactly 0.0 (safe to NaN)
        depth_crop = perspective_warp_nearest(depth_filled_bbox, depth_roi).astype(np.float32)
        depth_crop[depth_crop == 0] = np.nan

        # RGB
        rgb_path = analysis_dir / "averaged_images" / f"{run_name}_rgb_avg.png"
        rgb_img  = cv2.imread(str(rgb_path))
        rgb_crop = crop_to_roi(cv2.cvtColor(rgb_img, cv2.COLOR_BGR2RGB), rgb_roi) if rgb_img is not None else None

        # wall mask
        inner_mask = find_wall_mask(run_name, depth_crop.shape, depth_roi, analysis_dir)
        wall_mask = ~inner_mask

        min_depth = np.nanmin(depth_crop)
        processed = depth_crop - min_depth

        # 5. percentile clip to kill any residual spikes
        valid_vals = processed[inner_mask & ~np.isnan(processed)]
        if len(valid_vals) > 0:
            p_lo = np.percentile(valid_vals, 1)
            p_hi = np.percentile(valid_vals, 99)
            processed = np.clip(processed, p_lo, p_hi)

        if ultrasonic_level_mm is not None:
            ultrasonic_level_mm -= min_depth

        inner_raw = depth_crop.copy()
        inner_raw[wall_mask] = np.nan
        ground_truth_mm = np.nanmean(inner_raw) - min_depth

        wall_pct = 100.0 * wall_mask.sum() / wall_mask.size
        print(f"  Wall excluded: {wall_pct:.1f}%")
        if ultrasonic_level_mm is not None:
            print(f"  Ultrasonic (crimson): {ultrasonic_level_mm:.1f} mm")
        print(f"  Depth GT   (orange):  {ground_truth_mm:.1f} mm")
        if ultrasonic_level_mm is not None:
            print(f"  Diff (GT - US): {ground_truth_mm - ultrasonic_level_mm:+.1f} mm")

        # 2D plot
        fig, ax = plt.subplots(1, 2, figsize=(14, 6))
        if not np.all(np.isnan(processed)):
            im0 = ax[0].imshow(processed, cmap="viridis", vmin=0, vmax=global_max_depth)
            plt.colorbar(im0, ax=ax[0], label="Depth (mm)")
        else:
            ax[0].imshow(np.zeros_like(processed), cmap="viridis")
        ax[0].set_title(f"{run_name} - Depth (Zero-Shifted)")
        ax[0].axis("off")
        if rgb_crop is not None:
            ax[1].imshow(rgb_crop)
            ax[1].set_title(f"{run_name} - RGB ROI")
        else:
            ax[1].text(0.5, 0.5, "RGB not found", ha="center", va="center")
        ax[1].axis("off")
        plt.tight_layout()
        plt.show()

        # 3D plot
        h, w = processed.shape
        wall_z = processed.copy().astype(float)
        wall_z[inner_mask] = np.nan
        wall_surface = go.Surface(
            z=wall_z, x=np.arange(w), y=np.arange(h),
            colorscale=[[0,"rgba(120,120,120,0.6)"],[1,"rgba(120,120,120,0.6)"]],
            showscale=False, opacity=0.5, name="Excluded walls")

        planes = []
        small_ext, large_ext = 0.05, 0.10
        if ultrasonic_level_mm is not None:
            top_z = min(ultrasonic_level_mm, ground_truth_mm)
            ce = small_ext if ultrasonic_level_mm == top_z else large_ext
            oe = small_ext if ground_truth_mm      == top_z else large_ext
            planes.append(go.Surface(
                z=[[ultrasonic_level_mm]*2]*2,
                x=[0-ce*w, w-1+ce*w], y=[0-ce*h, h-1+ce*h],
                colorscale=[[0,"rgba(220,20,60,0.8)"],[1,"rgba(220,20,60,0.8)"]],
                showscale=False, opacity=0.8, name="Ultrasonic Reading (Crimson)"))
            planes.append(go.Surface(
                z=[[ground_truth_mm]*2]*2,
                x=[0-oe*w, w-1+oe*w], y=[0-oe*h, h-1+oe*h],
                colorscale=[[0,"rgba(255,140,0,0.8)"],[1,"rgba(255,140,0,0.8)"]],
                showscale=False, opacity=0.8, name="Ground Truth / Depth Map (Orange)"))
        else:
            planes.append(go.Surface(
                z=[[ground_truth_mm]*2]*2,
                x=[0-large_ext*w, w-1+large_ext*w], y=[0-large_ext*h, h-1+large_ext*h],
                colorscale=[[0,"rgba(255,140,0,0.8)"],[1,"rgba(255,140,0,0.8)"]],
                showscale=False, opacity=0.8, name="Ground Truth / Depth Map (Orange)"))

        fig = go.Figure(data=[
            go.Surface(z=processed, x=np.arange(w), y=np.arange(h),
                       colorscale="Viridis", cmin=0, cmax=global_max_depth,
                       colorbar=dict(title="Depth (mm)")),
            wall_surface] + planes)
        fig.update_layout(
            title=f"{run_name} - 3D Depth Analysis (Zero-Shifted)",
            scene=dict(
                xaxis_title="X (px)", yaxis_title="Y (px)", zaxis_title="Depth (mm)",
                xaxis=dict(range=[0, global_max_w]),
                yaxis=dict(range=[0, global_max_h]),
                zaxis=dict(autorange=False, range=[global_max_depth, 0]),
                aspectmode="manual",
                aspectratio=dict(x=global_max_w/global_max_depth,
                                 y=global_max_h/global_max_depth, z=1),
                camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))),
            width=700, height=700)
        fig.show()

    except Exception as e:
        print(f"Error processing {run_name}: {e}")
        import traceback; traceback.print_exc()


print("plot_run_analysis loaded.")


---\n## Run 1

In [ ]:
plot_run_analysis('run1', coordinates, run_dir)

In [140]:
plot_run_analysis('run1', coordinates, run_dir)

---\n## Run 2

In [ ]:
plot_run_analysis('run2', coordinates, run_dir)

---\n## Run 3

In [142]:
plot_run_analysis('run3', coordinates, run_dir)

---\n## Run 4

In [143]:
plot_run_analysis('run4', coordinates, run_dir)

---\n## Run 5

In [144]:
plot_run_analysis('run5', coordinates, run_dir)

---\n## Run 6

In [145]:
plot_run_analysis('run6', coordinates, run_dir)

---\n## Run 7

In [146]:
plot_run_analysis('run7', coordinates, run_dir)

---\n## Run 8

In [147]:
plot_run_analysis('run8', coordinates, run_dir)

---\n## Run 9

In [148]:
plot_run_analysis('run9', coordinates, run_dir)

---\n## Run 10

In [149]:
plot_run_analysis('run10', coordinates, run_dir)

---\n## Run 11

In [150]:
plot_run_analysis('run11', coordinates, run_dir)

---\n## Run 12

In [151]:
plot_run_analysis('run12', coordinates, run_dir)

---\n## Run 13

In [152]:
plot_run_analysis('run13', coordinates, run_dir)

---\n## Run 14

In [153]:
plot_run_analysis('run14', coordinates, run_dir)

---\n## Run 15

In [154]:
plot_run_analysis('run15', coordinates, run_dir)

---
## Run 16

In [155]:
plot_run_analysis('run16', coordinates, run_dir)


---
## Run 17

In [156]:
plot_run_analysis('run17', coordinates, run_dir)


---
## Run 18

In [157]:
plot_run_analysis('run18', coordinates, run_dir)


---
## Run 19

In [158]:
plot_run_analysis('run19', coordinates, run_dir)


---
## Run 20

In [159]:
plot_run_analysis('run20', coordinates, run_dir)


---
## Run 21

In [160]:
plot_run_analysis('run21', coordinates, run_dir)


---
## Run 22

In [161]:
plot_run_analysis('run22', coordinates, run_dir)


---
## Run 23

In [162]:
plot_run_analysis('run23', coordinates, run_dir)


---
## Run 24

In [163]:
plot_run_analysis('run24', coordinates, run_dir)


---
## Run 25

In [164]:
plot_run_analysis('run25', coordinates, run_dir)


In [165]:
plot_run_analysis('run25', coordinates, run_dir)


---
## Run 26

In [166]:
plot_run_analysis('run26', coordinates, run_dir)


---
## Run 27

In [167]:
plot_run_analysis('run27', coordinates, run_dir)


---
## Run 28

In [168]:
plot_run_analysis('run28', coordinates, run_dir)


---
## Run 29

In [169]:
plot_run_analysis('run29', coordinates, run_dir)


---
## Run 30

In [170]:
plot_run_analysis('run30', coordinates, run_dir)


---
## Run 31

In [171]:
plot_run_analysis('run31', coordinates, run_dir)


---
## Run 32

In [172]:
plot_run_analysis('run32', coordinates, run_dir)


---
## Run 33

In [173]:
plot_run_analysis('run33', coordinates, run_dir)


---
## Run 34

In [174]:
plot_run_analysis('run34', coordinates, run_dir)


---
## Run 35

In [175]:
plot_run_analysis('run35', coordinates, run_dir)


---
## Run 36

In [176]:
plot_run_analysis('run36', coordinates, run_dir)


---
## Run 37

In [177]:
plot_run_analysis('run37', coordinates, run_dir)


---
## Run 38

In [178]:
plot_run_analysis('run38', coordinates, run_dir)


---
## Run 39

In [179]:
plot_run_analysis('run39', coordinates, run_dir)


---
## Run 40

In [180]:
plot_run_analysis('run40', coordinates, run_dir)


---
## Run 41

In [181]:
plot_run_analysis('run41', coordinates, run_dir)


---
## Run 42

In [182]:
plot_run_analysis('run42', coordinates, run_dir)


---
## Run 43

In [183]:
plot_run_analysis('run43', coordinates, run_dir)


---
## Run 44

In [184]:
plot_run_analysis('run44', coordinates, run_dir)


---
## Run 45

In [185]:
plot_run_analysis('run45', coordinates, run_dir)


---
## Run 46

In [186]:
plot_run_analysis('run46', coordinates, run_dir)


---
## Run 47

In [187]:
plot_run_analysis('run47', coordinates, run_dir)


---
## Run 48

In [188]:
plot_run_analysis('run48', coordinates, run_dir)


---
## Run 49

In [189]:
plot_run_analysis('run49', coordinates, run_dir)


---
## Run 50

In [190]:
plot_run_analysis('run50', coordinates, run_dir)


---
## Bias Analysis by Base-Depth Group